# Korean Humanizer 시작하기
CPU 도구 확인 → 학습된 한국어 Humanizer 추론.

기본 실행은 **Qwen3-4B + 학습된 프로젝트 어댑터**입니다. 공개 승인 범위 확인 뒤 다운로드를 엽니다. 지금은 미공개 파일이 있으면 오류로 멈추며 기반 모델로 대체하지 않습니다.
[PC 설치](https://github.com/lollipop190000/korean-humanizer/blob/main/docs/INSTALL.md) · [Colab 안내](https://github.com/lollipop190000/korean-humanizer/blob/main/docs/COLAB.md)


## 1. CPU 도구 설치
GitHub 소스와 Python 3.11 환경을 준비합니다. 시스템 Python이나 Colab 기본 패키지는 바꾸지 않습니다.

In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.12.23"], check=True)
repo = Path.cwd() / "korean-humanizer"
if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/lollipop190000/korean-humanizer.git", str(repo)],
        check=True,
    )
else:
    print("기존 체크아웃을 재사용합니다:", repo)


def uv(*args):
    return subprocess.run([sys.executable, "-m", "uv", *args], cwd=repo, check=True)


uv("sync", "--frozen", "--python", "3.11", "--extra", "benchmark")
python = str(repo / ".venv" / "bin" / "python")


def run(*args):
    return subprocess.run([python, *args], cwd=repo, check=True)

## 2. GPU 없이 검증과 mock 실행
합성 자료를 사용한 소프트웨어 연결 검사이며, 모델의 문장 품질 평가가 아닙니다. 재실행할 때마다 출력 폴더를 새로 만듭니다.

In [ ]:
import tempfile

uv("run", "--frozen", "kh", "validate-sources", "tests/fixtures/sources.jsonl", "--allow-fixtures")
artifacts = repo / "artifacts"
artifacts.mkdir(exist_ok=True)
output_dir = Path(tempfile.mkdtemp(prefix="colab-", dir=artifacts)) / "mock"
uv(
    "run",
    "--frozen",
    "kh",
    "mock-pipeline",
    "tests/fixtures/sources.jsonl",
    "--output-dir",
    str(output_dir),
)
uv("run", "--frozen", "kh", "benchmark-validate", "eval/dev_seed.v1.jsonl")
print("결과 폴더:", output_dir)

## 3. 문장 입력과 설정 확인
설정 확인만 하며 모델 다운로드/추론은 하지 않습니다. 공유할 노트북에는 개인정보나 실제 말뭉치를 입력하지 마세요.

In [ ]:
TEXT = "회의는 내일 오후 3시에 시작합니다. 참석 인원은 12명입니다."
input_file = artifacts / "input.txt"
input_file.write_text(TEXT, encoding="utf-8")
run("scripts/rewrite_local.py", "--text-file", str(input_file))

## 4. 선택: 실제 학습 모델 추론
GPU 런타임으로 바꾼 뒤 1~3번 셀을 다시 실행하세요. 기본 `False`에서는 아무 모델도 내려받지 않습니다.
`True`로 바꾸면 추론 패키지 및 수 GB 모델을 다운로드합니다. GPU 배정은 Colab 계정 상황에 따라 달라집니다.
T4는 추론에 FP16을 사용하며 별도의 BF16 학습은 이 셀에서 수행하지 않습니다.

직접 보유한 호환 LoRA 폴더가 있다면 `ADAPTER_DIR`을 지정하세요. 기본값은 프로젝트 학습 모델입니다. 기반 모델만 비교하려면 `USE_BASE_ONLY = True`로 명시하세요.
처음 실행은 설치/다운로드로 오래 걸릴 수 있습니다. 출력의 `finish_reason`이 `length`면 글이 잘렸으므로 입력을 줄이거나 출력 한도를 조절하세요.


In [ ]:
RUN_MODEL = False
USE_BASE_ONLY = False
ADAPTER_DIR = ""  # 예: /content/my-adapter (config와 safetensors가 있는 폴더)
MAX_NEW_TOKENS = 256

if RUN_MODEL:
    # Colab 시스템 환경 대신 별도 추론 환경을 사용합니다.
    uv("venv", "--python", "3.11", "--allow-existing", ".venv-inference")
    infer_python = str(repo / ".venv-inference" / "bin" / "python")
    uv("pip", "install", "--python", infer_python, "-e", ".", "-r", "requirements/inference.txt")
    args = [
        infer_python,
        "scripts/rewrite_local.py",
        "--text-file",
        str(input_file),
        "--device",
        "cuda",
        "--max-new-tokens",
        str(MAX_NEW_TOKENS),
        "--execute",
    ]
    if USE_BASE_ONLY and ADAPTER_DIR:
        raise ValueError("기반 모델과 어댑터를 동시에 선택할 수 없습니다.")
    if USE_BASE_ONLY:
        args.append("--base")
    elif ADAPTER_DIR:
        args.extend(["--adapter", ADAPTER_DIR])
    subprocess.run(args, cwd=repo, check=True)
else:
    print("선택 추론은 꺼져 있습니다. GPU가 준비되면 RUN_MODEL = True로 바꾸세요.")

## 결과 보관
Colab 런타임 종료 시 파일이 사라질 수 있습니다. 필요한 파일은 왼쪽 파일 패널에서 내려받으세요.
노트북을 공유하기 전에 입력과 실행 출력을 지우세요. 기계 검사 통과는 사람의 의미·자연스러움 검수를 대신하지 않습니다.

검증 범위: 기본 CPU 셀과 dry-run. 실제 Colab GPU 추론은 이 안내 추가 작업에서 실행하지 않았습니다.
